In [1]:
# ############################################################
# 선형회귀 예시 — 매출(숫자) 예측 & MAE / RMSE / R2
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 불러오기 — 숫자/표 / 분리 / 모델 / 평가
# ------------------------------------------------------------
import numpy as np                                # 숫자 계산 (난수·제곱근)
import pandas as pd                               # 표 다루기
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # 선형회귀 (가장 잘 맞는 직선 찾기)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # 회귀 평가 3종

In [2]:
# ------------------------------------------------------------
# [목적] 예시용 판매 데이터 (매장크기·진열수·상품종류 -> 매출)
#        (실전에선 train.csv를 읽어 옵니다 / 상품종류는 글자 컬럼)
# ------------------------------------------------------------
rng = np.random.default_rng(0)                    # 난수 고정
n = 500                                           # 매장 500개
size = rng.normal(1500, 400, n).clip(200, 3000)   # 매장 크기
shelf = rng.integers(1, 20, n)                    # 진열대 수
kind = rng.choice(['음료','과자','생필품'], n)     # 상품 종류 (글자!)
eff = {'음료':300, '과자':0, '생필품':600}         # 종류별 매출 효과(정답 만들기용)
sales = size*0.6 + shelf*40 + np.array([eff[k] for k in kind]) + rng.normal(0, 300, n)  # 매출(정답)
train = pd.DataFrame({'매장크기': size.round(0), '진열수': shelf,
                      '상품종류': kind, '매출': sales.round(0)})   # 표로 묶기
train.head()                                      # 앞 5줄 확인

,매장크기,진열수,상품종류,매출
0,1550.0,17,음료,1813.0
1,1447.0,16,생필품,2228.0
2,1756.0,5,과자,731.0
3,1542.0,13,생필품,1914.0
4,1286.0,4,과자,887.0


In [3]:
# ------------------------------------------------------------
# [목적] 정답(y)/재료(X) 나누기 + 글자 -> 0/1 + 분할 (회귀는 stratify 없음)
# ------------------------------------------------------------
y = train['매출']                                 # 맞힐 값(숫자)
X = pd.get_dummies(train.drop(columns=['매출']))  # 글자 '상품종류'를 0/1 열로 (원핫)
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=0)          # 8:2 분할 (등급이 없어 stratify 불필요)

In [4]:
# ------------------------------------------------------------
# [목적] 학습(직선 찾기) -> 예측 -> 3가지 지표로 채점
# ------------------------------------------------------------
lr = LinearRegression().fit(X_tr, y_tr)           # 학습 = 기울기·절편 찾기
pred = lr.predict(X_val)                          # 예측(매출 숫자)

print('MAE :', round(mean_absolute_error(y_val, pred), 1))           # 평균 얼마나 틀렸나
print('RMSE:', round(np.sqrt(mean_squared_error(y_val, pred)), 1))   # 큰 오차에 더 민감한 점수
print('R2  :', round(r2_score(y_val, pred), 3))                      # 설명력(1=완벽, 0=평균수준)

MAE : 263.2
RMSE: 312.9
R2  : 0.671


In [5]:
# ------------------------------------------------------------
# [목적] (참고) 더 복잡한 모델로 바꿔보기 — 채점 코드는 그대로 재사용
# ------------------------------------------------------------
from lightgbm import LGBMRegressor                 # 부스팅 나무 모델(복잡·강력)
model = LGBMRegressor(n_estimators=500, learning_rate=0.05, verbose=-1)  # 나무 500그루
model.fit(X_tr, y_tr)                              # 학습
print('LGBM R2:', round(r2_score(y_val, model.predict(X_val)), 3))       # 선형회귀와 비교

LGBM R2: 0.524


In [6]:
# ============================================================
# [결과 해석]
#  · MAE ~ 263 -> 예측이 평균 263(매출 단위)만큼 빗나감
#  · RMSE ~ 313 (> MAE) -> 가끔 크게 틀린 예측이 있음 (RMSE는 큰 오차를 더 미워함)
#  · R2 ~ 0.671 -> 매출 변동의 약 67%를 설명 (그럭저럭)
#  · LGBM R2 ~ 0.524 < 선형 0.671 -> 이 데이터는 '직선' 관계라 오히려 단순한 선형회귀가 나음
#      (복잡한 모델이 항상 더 좋은 것은 아님!)
# ============================================================